# 16｜三站平日 08 時歷史站況

資料為 [CityBikes 2025 年一月歷史檔](https://data.citybik.es/dumps/by-network/2025/202501-capital-bikeshare-stats.parquet)。下載至 `data/raw/202501-capital-bikeshare-stats.parquet`，安裝 `duckdb` 後可重跑。站況採集時間由 UTC 換算為華盛頓當地時間。旅次檔暫按當地時間解讀。

站況為變更紀錄。08:00–08:59 內只要有一筆零值，即列為「觀測到零」。若所有觀測皆為正值，且相鄰紀錄以及時段兩端到最近紀錄的間隔都不超過 10 分鐘，列為「紀錄內未見零」。其餘列為「涵蓋不足」。10 分鐘是本次的判讀門檻，另用 5 和 15 分鐘比較結果。即使符合門檻，採集間隔內仍可能出現短暫零值。可判讀天數包含前兩種結果。

Bike-share data by CityBikes contributors and available from [CityBikes](https://data.citybik.es).


In [1]:
import os
from pathlib import Path
from zipfile import ZipFile

import duckdb
import pandas as pd
from IPython.display import display

archive = Path(os.environ.get(
    'CITYBIKES_PARQUET', '../data/raw/202501-capital-bikeshare-stats.parquet'))
stations = {
    '31600': ('5th & K St NW', 'bikes'),
    '31101': ('14th & V St NW', 'bikes'),
    '31699': ('Eastern Market Metro', 'free'),
}
with duckdb.connect() as db:
    snapshots = db.execute('''
        SELECT id, name, latitude, longitude, timestamp, bikes, free
        FROM read_parquet(?)
        WHERE tag = 'capital-bikeshare' AND name IN (?, ?, ?)
        ORDER BY timestamp
    ''', [str(archive), *(name for name, _ in stations.values())]).df()
snapshots['local_time'] = (snapshots.timestamp.dt.tz_localize('UTC')
                           .dt.tz_convert('America/New_York').dt.tz_localize(None))

with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        trips = pd.read_csv(f, dtype='string')
dates = pd.date_range('2025-01-01', '2025-01-31')
weekdays = dates[(dates.dayofweek < 5) & ~dates.day.isin([1, 9, 20])]
assert len(weekdays) == 20

def verdict(minimum, gap, limit):
    if minimum == 0:
        return '觀測到零'
    if pd.notna(minimum) and gap <= limit:
        return '紀錄內未見零'
    return '涵蓋不足'

rows = []
for station_id, (name, metric) in stations.items():
    station = snapshots.loc[snapshots.name.eq(name)]
    assert station.id.nunique() == 1
    assert station[['latitude', 'longitude']].drop_duplicates().shape[0] == 1
    morning_counts = {}
    for endpoint, time_field, direction in (
        ('start', 'started_at', '借出'), ('end', 'ended_at', '歸還')
    ):
        matches = trips.loc[trips[f'{endpoint}_station_id'].str.strip().eq(station_id)]
        assert matches[f'{endpoint}_station_name'].dropna().eq(name).all()
        for coordinate, short in [('latitude', 'lat'), ('longitude', 'lng')]:
            median = matches[f'{endpoint}_{short}'].astype(float).median()
            assert abs(median - station[coordinate].iloc[0]) < 0.0001
        times = pd.to_datetime(matches[time_field], format='mixed', errors='raise')
        morning_counts[direction] = (times[times.dt.hour.eq(8)].dt.normalize().value_counts()
                                     .reindex(weekdays, fill_value=0))

    for date in weekdays:
        start = date + pd.Timedelta(hours=8)
        stop = start + pd.Timedelta(hours=1)
        records = station.loc[station.local_time.ge(start) & station.local_time.lt(stop)]
        usable = records.dropna(subset=[metric])
        points = pd.DatetimeIndex([start, *usable.local_time, stop])
        gap = (points[1:] - points[:-1]).max().total_seconds() / 60
        minimum = usable[metric].min() if len(usable) else float('nan')
        zero_times = usable.loc[usable[metric].eq(0), 'local_time']
        starts = morning_counts['借出'].at[date]
        ends = morning_counts['歸還'].at[date]
        rows.append({'站點': station_id, '日期': date.strftime('%m/%d'),
                     '指標': '可借車' if metric == 'bikes' else '可還空位',
                     '借出': starts, '歸還': ends, '借還差額': starts - ends,
                     '有效紀錄': len(usable), '最大間隔（分鐘）': gap, '最低觀測值': minimum,
                     '首筆零值時間': zero_times.iloc[0].strftime('%H:%M:%S') if len(zero_times) else '',
                     '判讀': verdict(minimum, gap, 10)})

daily = pd.DataFrame(rows)
assert len(daily) == 60 and not daily.duplicated(['站點', '日期']).any()
assert daily.groupby('站點')['借還差額'].sum().to_dict() == {'31101': 101, '31600': 126, '31699': -95}
summary = (pd.crosstab(daily['站點'], daily['判讀'])
           .reindex(index=list(stations), columns=['觀測到零', '紀錄內未見零', '涵蓋不足'], fill_value=0))
summary['可判讀天數'] = summary['觀測到零'] + summary['紀錄內未見零']
assert (summary['可判讀天數'] + summary['涵蓋不足']).eq(20).all()
sensitivity = pd.DataFrame({
    limit: daily.apply(lambda row: verdict(row['最低觀測值'], row['最大間隔（分鐘）'], limit)
                       != '涵蓋不足', axis=1).groupby(daily['站點']).sum()
    for limit in (5, 10, 15)
}).loc[list(stations)]
display(summary)
print('各間隔門檻下的可判讀天數')
display(sensitivity)
display(daily.round({'最大間隔（分鐘）': 1}))

各間隔門檻下的可判讀天數


判讀,觀測到零,紀錄內未見零,涵蓋不足,可判讀天數
站點,,,,
31600,2,3,15,5
31101,1,1,18,2
31699,1,2,17,3


,5,10,15
站點,,,
31600,2,5,13
31101,1,2,7
31699,1,3,10


,站點,日期,指標,借出,歸還,借還差額,有效紀錄,最大間隔（分鐘）,最低觀測值,首筆零值時間,判讀
0,31600,01/02,可借車,5,2,3,7,12.0,5.0,,涵蓋不足
1,31600,01/03,可借車,5,3,2,9,20.3,1.0,,涵蓋不足
2,31600,01/06,可借車,0,0,0,0,60.0,NaN,,涵蓋不足
3,31600,01/07,可借車,0,2,-2,2,26.5,8.0,,涵蓋不足
4,31600,01/08,可借車,3,1,2,5,27.0,0.0,08:45:31,觀測到零
5,31600,01/10,可借車,9,3,6,9,21.0,6.0,,涵蓋不足
6,31600,01/13,可借車,9,1,8,9,12.0,3.0,,涵蓋不足
7,31600,01/14,可借車,15,1,14,12,21.0,0.0,08:58:35,觀測到零
8,31600,01/15,可借車,13,2,11,16,9.0,4.0,,紀錄內未見零
9,31600,01/16,可借車,11,3,8,15,7.5,5.0,,紀錄內未見零


## 31600 的兩天時間線

查核 2025 年 1 月 8 日與 14 日的 07:30–09:30。若窗口內缺少前置站況，向前取最近一筆。用站況採集時間對齊旅次，各區間排除起點並包含終點。差額為實際觀測的可借車變化減去旅次可解釋的變化（歸還減借出）。旅次檔暫按華盛頓當地時間解讀。採集間隔與時間偏差會影響對帳，差額原因須另查營運紀錄。

In [2]:
target = '31600'
status = snapshots.loc[snapshots.name.eq(stations[target][0]) & snapshots.bikes.notna()]
station_trips = {
    direction: pd.to_datetime(
        trips.loc[trips[f'{end}_station_id'].str.strip().eq(target), time_field],
        format='mixed', errors='raise')
    for end, time_field, direction in (
        ('start', 'started_at', '借出'), ('end', 'ended_at', '歸還'))
}

def interval(label, first, last):
    start, stop = first.local_time, last.local_time
    counts = {direction: times.between(start, stop, inclusive='right').sum()
              for direction, times in station_trips.items()}
    change = int(last.bikes - first.bikes)
    return {'區間': label, '開始': start.strftime('%H:%M:%S'),
            '結束': stop.strftime('%H:%M:%S'), '起點可借車': first.bikes,
            '終點可借車': last.bikes, **counts, '庫存變化': change,
            '對帳差額': change - counts['歸還'] + counts['借出']}

comparisons = []
comparison_endpoints = []
details = []
for day in ('2025-01-08', '2025-01-14'):
    morning = pd.Timestamp(day) + pd.Timedelta(hours=8)
    cutoff = morning + pd.Timedelta(minutes=90)
    observed = status.loc[status.local_time.between(
        morning - pd.Timedelta(minutes=30), cutoff)].reset_index(drop=True)
    before = status.loc[status.local_time.lt(morning)].iloc[-1]
    zero = observed.loc[observed.local_time.ge(morning)
                        & observed.local_time.lt(morning + pd.Timedelta(hours=1))
                        & observed.bikes.eq(0)].iloc[0]
    last_positive = status.loc[status.local_time.lt(zero.local_time)
                                 & status.bikes.gt(0)].iloc[-1]
    recovery = observed.loc[observed.local_time.gt(zero.local_time) & observed.bikes.gt(0)]
    comparison_endpoints.extend([
        (day, '08 時前最近紀錄至首筆零值', before, zero),
        (day, '零值前最後有車至首筆零值', last_positive, zero),
    ])
    comparisons.extend([
        {'日期': day, **interval('08 時前最近紀錄至首筆零值', before, zero)},
        {'日期': day, **interval('零值前最後有車至首筆零值', last_positive, zero)},
    ])
    if recovery.empty:
        comparison_endpoints.append((day, '截至 09:30 未觀測到恢復', zero, None))
        comparisons.append({
            '日期': day, '區間': '截至 09:30 未觀測到恢復',
            '開始': zero.local_time.strftime('%H:%M:%S'), '結束': cutoff.strftime('%H:%M:%S'),
            '起點可借車': zero.bikes,
            **{direction: times.between(zero.local_time, cutoff, inclusive='right').sum()
               for direction, times in station_trips.items()},
        })
    else:
        comparison_endpoints.append((day, '首筆零值至下一筆有車', zero, recovery.iloc[0]))
        comparisons.append({'日期': day, **interval('首筆零值至下一筆有車', zero, recovery.iloc[0])})
    for i in range(1, len(observed)):
        details.append({'日期': day, **interval('相鄰站況', observed.iloc[i-1], observed.iloc[i])})
timeline = pd.DataFrame(comparisons)
display(timeline)
display(pd.DataFrame(details))

,日期,區間,開始,結束,起點可借車,終點可借車,借出,歸還,庫存變化,對帳差額
0,2025-01-08,08 時前最近紀錄至首筆零值,07:12:33,08:45:31,2,0.0,3,1,-2.0,0.0
1,2025-01-08,零值前最後有車至首筆零值,08:42:32,08:45:31,1,0.0,0,0,-1.0,-1.0
2,2025-01-08,首筆零值至下一筆有車,08:45:31,09:21:32,0,1.0,0,1,1.0,0.0
3,2025-01-14,08 時前最近紀錄至首筆零值,07:55:37,08:58:35,13,0.0,14,2,-13.0,-1.0
4,2025-01-14,零值前最後有車至首筆零值,08:55:36,08:58:35,2,0.0,1,0,-2.0,-1.0
5,2025-01-14,截至 09:30 未觀測到恢復,08:58:35,09:30:00,0,NaN,2,1,NaN,NaN


,日期,區間,開始,結束,起點可借車,終點可借車,借出,歸還,庫存變化,對帳差額
0,2025-01-08,相鄰站況,08:09:32,08:36:32,1,2,1,1,1,1
1,2025-01-08,相鄰站況,08:36:32,08:39:31,2,1,0,0,-1,-1
2,2025-01-08,相鄰站況,08:39:31,08:42:32,1,1,1,0,0,1
3,2025-01-08,相鄰站況,08:42:32,08:45:31,1,0,0,0,-1,-1
4,2025-01-08,相鄰站況,08:45:31,09:21:32,0,1,0,1,1,0
5,2025-01-08,相鄰站況,09:21:32,09:24:32,1,1,0,1,0,-1
6,2025-01-08,相鄰站況,09:24:32,09:27:32,1,2,0,0,1,1
7,2025-01-14,相鄰站況,07:34:33,07:37:36,17,16,0,0,-1,-1
8,2025-01-14,相鄰站況,07:37:36,07:40:35,16,16,1,0,0,1
9,2025-01-14,相鄰站況,07:40:35,07:43:35,16,15,0,0,-1,-1


## 採集時間與來源更新時間的對帳比較

固定原時間線的站況端點和可借車數，分別以 CityBikes 採集時間與來源欄位 `extra.last_updated` 切分旅次。兩種口徑均排除起點、包含終點，時間換算至華盛頓當地時間，旅次時間暫依當地時間解讀。來源欄位轉換自 Unix 秒，代表站況所附的更新時間。實際借還和站況變動可能在不同時間寫入。

檢查每天 07:00–09:30 內的更新時間是否缺漏、晚於採集時間或比上一筆更新時間更早。09:30 為固定查核截止時間，無恢復站況時只比較旅次件數。

In [3]:
with duckdb.connect() as db:
    updates = db.execute('''
        SELECT timestamp, TRY_CAST(extra->>'last_updated' AS BIGINT) AS update_epoch
        FROM read_parquet(?)
        WHERE tag = 'capital-bikeshare' AND id = ?
          AND (timestamp BETWEEN '2025-01-08 12:00' AND '2025-01-08 14:30'
            OR timestamp BETWEEN '2025-01-14 12:00' AND '2025-01-14 14:30')
        ORDER BY timestamp
    ''', [str(archive), status.id.iloc[0]]).df()
updates['採集時間'] = (updates.timestamp.dt.tz_localize('UTC')
                        .dt.tz_convert('America/New_York').dt.tz_localize(None))
updates['來源更新時間'] = (pd.to_datetime(updates.update_epoch, unit='s', utc=True, errors='coerce')
                            .dt.tz_convert('America/New_York').dt.tz_localize(None))
updates['日期'] = updates['採集時間'].dt.date
updates['缺漏'] = updates['來源更新時間'].isna()
updates['晚於採集'] = updates['來源更新時間'].gt(updates['採集時間'])
updates['較前筆倒序'] = updates.groupby('日期')['來源更新時間'].diff().lt(pd.Timedelta(0))
flags = ['缺漏', '晚於採集', '較前筆倒序']
display(updates.groupby('日期').agg(紀錄數=('採集時間', 'size'), **{
    flag: (flag, 'sum') for flag in flags}))
display(updates.loc[updates[flags].any(axis=1), ['採集時間', '來源更新時間', *flags]])
source_times = updates.set_index('採集時間')['來源更新時間']
assert source_times.index.is_unique

results = []
boundary_trips = []
for day, label, first, last in comparison_endpoints:
    start = first.local_time
    stop = last.local_time if last is not None else pd.Timestamp(day + ' 09:30')
    updated_start = source_times.get(start, pd.NaT)
    updated_stop = source_times.get(stop, pd.NaT) if last is not None else stop
    change = int(last.bikes - first.bikes) if last is not None else None
    row = {'日期': day, '區間': label, '庫存變化': change}
    masks = {}
    for clock, left, right in [('採集', start, stop), ('更新', updated_start, updated_stop)]:
        row[f'{clock}起點'] = left
        row[f'{clock}終點'] = right
        if pd.isna(left) or pd.isna(right) or left >= right:
            continue
        masks[clock] = {direction: times.between(left, right, inclusive='right')
                        for direction, times in station_trips.items()}
        counts = {direction: mask.sum() for direction, mask in masks[clock].items()}
        row.update({f'{clock}{direction}': count for direction, count in counts.items()})
        if change is not None:
            row[f'{clock}差額'] = change - counts['歸還'] + counts['借出']
    if '更新' in masks:
        for direction, times in station_trips.items():
            added = masks['更新'][direction] & ~masks['採集'][direction]
            removed = masks['採集'][direction] & ~masks['更新'][direction]
            for action, mask in [('納入', added), ('排除', removed)]:
                for index, time in times[mask].items():
                    boundary_trips.append({'日期': day, '區間': label, '方向': direction,
                                           '變動': action, '旅次 ID': trips.at[index, 'ride_id'],
                                           '事件時間': time})
    results.append(row)

clock_comparison = pd.DataFrame(results)
for new, old in [('採集借出', '借出'), ('採集歸還', '歸還'),
                 ('採集差額', '對帳差額'), ('庫存變化', '庫存變化')]:
    assert clock_comparison[new].astype('Float64').equals(timeline[old].astype('Float64'))
display(clock_comparison)
display(pd.DataFrame(boundary_trips, columns=['日期', '區間', '方向', '變動', '旅次 ID', '事件時間']))

,紀錄數,缺漏,晚於採集,較前筆倒序
日期,,,,
2025-01-08,10,0,0,0
2025-01-14,21,0,0,0


,採集時間,來源更新時間,缺漏,晚於採集,較前筆倒序


,日期,區間,庫存變化,採集起點,採集終點,採集借出,採集歸還,採集差額,更新起點,更新終點,更新借出,更新歸還,更新差額
0,2025-01-08,08 時前最近紀錄至首筆零值,-2.0,2025-01-08 07:12:33,2025-01-08 08:45:31,3,1,0.0,2025-01-08 07:11:44,2025-01-08 08:44:46,3,1,0.0
1,2025-01-08,零值前最後有車至首筆零值,-1.0,2025-01-08 08:42:32,2025-01-08 08:45:31,0,0,-1.0,2025-01-08 08:41:21,2025-01-08 08:44:46,1,0,0.0
2,2025-01-08,首筆零值至下一筆有車,1.0,2025-01-08 08:45:31,2025-01-08 09:21:32,0,1,0.0,2025-01-08 08:44:46,2025-01-08 09:20:56,0,1,0.0
3,2025-01-14,08 時前最近紀錄至首筆零值,-13.0,2025-01-14 07:55:37,2025-01-14 08:58:35,14,2,-1.0,2025-01-14 07:53:33,2025-01-14 08:57:53,14,2,-1.0
4,2025-01-14,零值前最後有車至首筆零值,-2.0,2025-01-14 08:55:36,2025-01-14 08:58:35,1,0,-1.0,2025-01-14 08:53:50,2025-01-14 08:57:53,2,0,0.0
5,2025-01-14,截至 09:30 未觀測到恢復,NaN,2025-01-14 08:58:35,2025-01-14 09:30:00,2,1,NaN,2025-01-14 08:57:53,2025-01-14 09:30:00,2,1,NaN


,日期,區間,方向,變動,旅次 ID,事件時間
0,2025-01-08,零值前最後有車至首筆零值,借出,納入,（重跑可查 ID）,2025-01-08 08:41:22.028
1,2025-01-14,零值前最後有車至首筆零值,借出,納入,（重跑可查 ID）,2025-01-14 08:55:32.365
